# Batch norm and dropout from scratch

Build the two layers whose behavior depends on training versus eval mode, from `nn.Parameter` and `register_buffer`,
so that they match `nn.BatchNorm1d` and `nn.Dropout` in both modes.

```python
class MyBatchNorm1d(nn.Module):
    def __init__(self, num_features, eps=1e-5, momentum=0.1):
        ...
    def forward(self, x):
        ...

class MyDropout(nn.Module):
    def __init__(self, p=0.5):
        ...
    def forward(self, x):
        ...
```

Inside `forward`, `self.training` is `True` after `.train()` (the default for a new module) and `False` after `.eval()`.

**MyBatchNorm1d.** `x` is a float tensor of shape `(N, C)` with `C = num_features` and $N \ge 2$ in training mode.

- Parameters `weight` ($\gamma$, ones) and `bias` ($\beta$, zeros), shape `(C,)`, and buffers `running_mean` (zeros),
  `running_var` (ones), both shape `(C,)`, and `num_batches_tracked` (`torch.tensor(0)`, dtype `torch.long`). Use exactly
  these names, so that `state_dict()` has the same keys as `nn.BatchNorm1d`'s.
- **Training mode:** for each column, compute the batch mean $\mu$ and the **biased** variance $\sigma^2$ (divide by $N$), and
  return $\gamma\,(x - \mu)/\sqrt{\sigma^2 + \epsilon} + \beta$. Gradients must flow to `x`, `weight` and `bias` through these batch
  statistics. Then update the buffers without recording gradients:
  `running_mean` $\leftarrow (1 - m)\,$`running_mean` $+ m\mu$ and
  `running_var` $\leftarrow (1 - m)\,$`running_var` $+ m\,s^2$, where $s^2$ is the **unbiased** variance (divide by $N - 1$) and
  $m$ is `momentum`; add 1 to `num_batches_tracked`.
- **Eval mode:** return $\gamma\,(x - \text{running\_mean})/\sqrt{\text{running\_var} + \epsilon} + \beta$ and change no buffer.

**MyDropout.** `x` is a float tensor of any shape; `0 <= p < 1`. No parameters.

- **Training mode:** set each entry to 0 independently with probability `p` and multiply the others by $1/(1 - p)$.
  Use `torch.rand_like` or `torch.bernoulli` for the randomness (a new mask on every call).
- **Eval mode:** return `x` unchanged.

**Example.**

```python
bn = MyBatchNorm1d(1)
bn(torch.tensor([[2.0], [4.0], [4.0], [4.0], [5.0], [5.0], [7.0], [9.0]])).squeeze(1)
# tensor([-1.5000, -0.5000, -0.5000, -0.5000,  0.0000,  0.0000,  1.0000,  2.0000], grad_fn=...)
bn.running_mean, bn.running_var          # tensor([0.5000]), tensor([1.3571])
bn.eval()
bn(torch.tensor([[5.0]]))                # tensor([[3.8628]], grad_fn=...)

drop = MyDropout(0.25)
y = drop(torch.ones(8))                  # e.g. tensor([1.3333, 0.0000, 1.3333, 1.3333, ...])
drop.eval()
drop(torch.ones(8))                      # tensor([1., 1., 1., 1., 1., 1., 1., 1.])
```

**Restrictions.** Only `import torch` and `import torch.nn as nn`. Do not use PyTorch's batch norm, layer norm or dropout
in any form: the grader bans the names `BatchNorm1d`, `BatchNorm2d`, `batch_norm`, `LayerNorm`, `layer_norm`,
`instance_norm`, `Dropout`, `dropout` and `functional` (so do not name a variable `dropout` either). The batch statistics
must be computed from `x` with ordinary tensor operations (`mean`, `var`, `sum`, ... are allowed).

**Constraints.** $N \le 4096$, $C \le 512$, `eps` in $[10^{-6}, 10^{-3}]$, `momentum` in $(0, 1)$. Time limit 1 s per forward
call, CPU. Outputs must match PyTorch to about $10^{-5}$.


## How to use this notebook

1. Write your solution in the next cell. Running it saves the file `problem4_submission.py`, the same way the contest's Colab notebook does.
2. Run the grader cell at the bottom. It runs the example and the hidden tests and checks the import and ban rules.
3. Edit, re-run the solution cell, then re-run the grader.

From the USAAIO Prep study site (https://bengreff.github.io/usaaio-prep/); not affiliated with or endorsed by USAAIO.

In [ ]:
%%writefile problem4_submission.py
import torch
import torch.nn as nn


class MyBatchNorm1d(nn.Module):
    def __init__(self, num_features, eps=1e-5, momentum=0.1):
        super().__init__()
        # Your code

    def forward(self, x):
        # Your code
        return out


class MyDropout(nn.Module):
    def __init__(self, p=0.5):
        super().__init__()
        # Your code

    def forward(self, x):
        # Your code
        return out


Optional: import it the way the grader does and try the example from the statement.

In [ ]:
import importlib
import problem4_submission as submission
importlib.reload(submission)
# e.g. submission.MyBatchNorm1d(...)

## Grade

In [ ]:
#@title Grader: run this cell to grade your solution { display-mode: "form" }
# The grading code is hidden on purpose (double-click to see it). It writes the site's test harness and this
# problem's grader to a temporary folder, imports your file the way the contest grader does, checks the import
# and ban rules, runs every test (including hidden ones) and prints a table.
import ast, base64, contextlib, importlib.util, io, json, os, signal, sys, time, traceback, zlib
_P = json.loads(zlib.decompress(base64.b64decode("eNq9XItz2zaa/1ew7syGzMqM5MR9OE3v0tTt5iavSbKze2N7WEqEZK4pkCXIyKrX97ff9/sAkCBFOY/uXTpjSSDwvfC9AfbmIEsPTsRBM5sdltNHBxNxsJZ1QkM3u0/qrM4lxn5M6sWlUEW1FolKRVoVZdHUYlkVa6EXFZ5ifi61LpQDkk+/Y/CZamqpafTomH6m2XKZLZq83tLIIxqQqq7w/eDlltG8IiyzlFcWaZPLWCVrJqKsinku149i3czXmdYZoaJZSZ4XG5nG2bosqhp4zg7qoiKKLujpPFFKpjw4gN7+POKfc/yMwSJ+/WQ4xNe0+7ps1KImtEnO3CZbWbUrMqXrRC1kO/ACjwHf0RHnRVGCvmWSawnxSl3HdbaWBD7Gg4dTjCYrw4RSERPlANJvjxZmMTIi6n7Pm+VSVtpOr6skU4fyA9ELGnTRVAsWZVFlq8yyUSZVfUSDddXIWzCZ5bxbNweXSaVoR6OSt+ec//urzEtCIOrLpBZ2R8SqSlIMmi0Qwa+sGHa5G73/axidq3P11E4XmRaLQtW0/w8aldX6wff4+OGBhUq/yyz94YGZTURE4nktUrnMCKqA7ITbDy2gIykPxvcNbXVyJembPFeduggjLvErDf0aidN/PH355sXpO5FnugZL0oAFMC30ZbFRgha9bZQIPmQ6I6rCx+IdwNWiagit/CCrLS8i1n5OsrypaCVJXUuBn2KT1ZciEQRPJyspNlVWE8NiWVTiR6lOhE62YgNyN4mmp4VaTYQCVJE261JcZmlK0zNVNrWG9MwuKCvSRVFu2x9QJMjX/lS0fisIqiox+pV4VuTJXDx78zctkkoKDaupICsFUu5p8fJIvKmKidCFyItFkgNiplZarBsSylxCqCSiAvab0eM8gxzmxP8lbeUyWZAKRsB0FE2xuYlYNdj/wGDWBCuXh/VlJZNU6FLShtGsqmhWl/mWtCVfknMBFYuigqArCaHXIluCQPrJ9m35sGYDobx7c3r6U/zz02fvX78VT4AdDJ+rRZ4QdmxDcHq9kCU0JTw5V4L+lfTIzCKFIonLNCBdTCdirVduDuFVBYSsUjuCf93mBpjbAYlVGVy7teTSvDVuh2Ci3uiS+HduI7iemOfRe6l0UYXeckYr66ZS4jpKyVsvLoMwWpQN/eVtDkIzWTKb4jmjO62qovKgGJY9WKqMEp1UVbIlujs+9GVSymBV1BNSSoW/pJ9Pzg9Is8lrnx84yjTJum7KXAbgnOaHES+1tLBMaY6bBFjhRCzPD24A8JY2XBtc4kaT35HXpVzUpBM33vxbQtdRtsgLfRdlE1HVRf5kJg+/nojEfv3W0UuWtSGSW2ITXW+J0WVeJCAM44yz/8BjZmX4A0sb83UPO4YBOz0c4809YvZaBLQhmSb3ltW0mujI8yAU5Cmghf6zjX3mo4cnJWcPT6iwJlPLFrhVZOx3nlshkjCsuPDHigt/fMWTVUUSY0GAuHVyjY9kroOVOBSbMAwBmxjNfpdCUlATUxjfiKn05cSOTlBYIv8ggjypVnC7EgorbujjJHq4ug17ey9/a5L8E7Syb1RmKohWKWv6JGSJEhFJXVf2+fkBb0cHZERdPAXp5vyfq4WvGkx/bOVAxO0Yky/VFgJ25WRIMSkDqGXmOxjkjG/o0Z+qHmmYRUO9zch0zCbC24Fveu9+MEJvR7olO/RjlERhAYdRzJlfHPfIWZH7w85ZrPfcpHsOap/SWKukpEhe//+65rxQ8jOdMqF0WnkNleTYfhByvg3r7dMTpNmCBInEJfTp6TPXow4Ag1TK8sl7yvQ8BbMEvifxDcnbBRD2QgiGIsDkZ9c9wdO2BMlEzL9c7sknyd1MMXbB+L5U7AnEznB03xVYRIlBooN5OOZraHnnZ0aXt35ibr/y3vYtGxxMjLcjzVb+Xjk4DMFLXop43ZBRUHYTLNVE3Cd3SpZx//7VBt8cJSZ1fEYRQCwVY14iQQUPlONQFrgi80oUpYzVqllTUk4hRJR5spCRyzo56sOeyC+edZaVhJzPJlgAjBdmJgpECh4qGBJkHmNJRopMDOsQSyWlMpJKSRn8npWBWcM4egrO/qRVLutEtlTYkC6mEgl+tsyQWSJl7nj56iYTfxGz28diU1RXyOoTVl6B7aN8tPWXVshEfCdhJPGZCrREKkhUfaqUH/siJu/qp9wjmTQXshQPNeXexAEExLnuvXYlJe+mhgKq1wSiQglGBdJagyXOw18mC7EgBFhCTJqsG9UKJ/ZIDqi6OJw3KXlSDHMOTml5hgKqKcmcRL3JFhIASUIZRehiaS2YaHYS1axCGFlSvoTwDQi0oqFnkXiPB1lFw8RMfg8oEBcMMkhYpj21qqfIJglCRNXlMmYwsnLe5iOqlNbjiylHqW06Yvl9IuwmigfCLxs6lYyhihWsITjy9Y42kdB8/8RCGriiOdU0V91QcjQRV0eEre8fmeTJYLDHycdFwXRCEEcshiNvmMVAFVuQIsTuEYefb/oMsR3VRcF6Q8V/Wp9ER8tbyvJJy+RjW3W1SszKRGbHgESjUM/fWNHYdVxDUmHntnjkH6OkypLK4CWp1goJrCkyb+w+MSBYx50GWqkVWadM3YZ1BQ7tZFqsSdrLhNQvbieec+uo0UmSFTF3GHpdjl8wQhZAlHXNg4ydxmWWp7C6BTojVDBnsAXM/K93r18RrYotR9cpUci2Wm7rSxo5fC4cHvG9bXHEaVb9IL7vcNBD+k1W+S8eq3/YKfnJ2Lzynzb2us6zeTtkPmgkauos74aL9us/ddE1CcqkvvSX62ylkm6Z3upBe8H9qCguzJPFVTtCWqzQLDC9Br3IiE0lW3dHA1dZfZhLmiZm0dfRzOZ0+gR9ojU6hxN2KVRd2ODjnF+ZZCTl1blyOKJllpNGu5/B+QGRXVQSFaDttFAmGt3/m7pS6OHoIkdHpeAOACHMSuxadN9min89fXtKlmNlEb2hzyCO0QaL4zAix4XllNWVSUWUnSuSSoTJEWmrrOpgSsZA6QPAhARwrPfFvSDxFcXq35ITcfpoesSdsBd/f/rf7+Ifn756dfoTUXBzfhDbNmYcgxlizO2n+Smv5cJ+ozzffFsU65JoPT+4PVcvXr9+E796/dPpOwIXkK5EPxdkq/jyd1Jcab4+1Vu1aB+8oBzyGcEwv95J78dPlGR2v36RCvG5qE6vy9Bvsbw3vZiPdVnihCq9dQAda9bkbyhsydZiuVR0gPxuBDKbRby4lIsrHehqgS2uk368fWvsHdkwBSvIsbgywlnrFQnmMblFZICmx8dmo+t77e5wMmRbiQhUdZWZtmIvRJkgdHYxls3WlaTEjqVEWkL1NRHaz0Pfbak+v+Y8FE05uZsfnlmyT0hbuD9sqT+Bm9S83JbIhTJ+5kZG+FTF7Qm+02wqfm4tgbYtjpBA8orIMZOdDHrlkNHZhSV0nrTjHCdrF3K4HRo3moF543Oyh9j0Su04VJxSRrLdLCVTQSZWmUTBACYIZn6AigZJdzbH2QCxc8W5m8wjKLZFwP1yT+brpF43uT+y5O4IuWr4ZhL+JsmvAmzGIHR7aTqmG302pcGwqmhTWUyMmNyT3QBmWyqULwcJT0JSvkZrwfyMCBsJEpljsKbMk9wRijkAXzOxZiPCk/HY6O1FlKSpxRHuTu72xpsX6TLPsN3AeXY4uxhZ2G1e1JQpsu6RtX6RmN8txp/J6w25oQ2lvWI52sY7sd8Z1IgseYUVJH3/A1IcSpCgDaTgia4ngTENGCzdJ7wvWsp8fq7QnzrbGfI+UAimA4b2SVBfeR75LoBZup+r8UljGLtgNcTJhh8lZSlVGriOj1l0fmBcnnHv/9GX1xgWjoI/Zuq1jWJPm9VTjRgUmt7KYEFU2Hkvk/olpYw7Om180GcSR4R5CumBRJJoQe11/y4LMFZiVB6jbBCYF/2zQHnKDjbw8IThuHf5CyIKCcaBurk3EfcMEGdRMMB7hI/q3tU9tGJvx/uJY/SjZTFCvna0d8C+ak8Q54nSkC1OmTlCcCrYBQhuP8uEith/8Gf42FSj+SbZ6kNzyEuVDOWFWQsISmmhFSrfThxOqly6UBMYLTBNJJtKIenYilyuqDTfIG0SPDOKotALZpSUibTAphjAlxnXl2YXAi9o/pmDoxeFDbk2+JIa/ksEXiz9s+jlhZ4KEYbPVh2iQAsrH8APdDiiOwQ5/CM7rAqHoz2S7XaZKO8zHxsjiovS5JP9bhumGyMbmN6nMIyKkvTiP4WDjZLVsxsRmMzJYjibXvjKPcL+54igw3oX8+zcdljm0S9j2KRJrOw9RmkYZgylruSlVFzEtrkjr/r38m85u72jUl9WGYHJtzFnsgGBoEI3RjXVz+hfK3nIdKZSL6qMqwpRkqqSfa+EPYnmCZk7vkbBRuYqMU/zYX2/wzRHW8XVrRHVzfS9jut5IKM4bh9QyWeTTQCnTHPJQX0JPASCdmqJ8lO6nA+VX8uCzUo3aJmgYQUvy71Jq3T0F+mYl7X3hrAVrDCYzcdpHflWmDgx4RMS6Z2PIPu/vWGst5jvhL2mTDxoazHytMMqF5VsUq0+nLUpomNl78yjC68ktpJCAv5EtFMeunSd9J7G0XCI8iJJdRAwEQ+gNrAJbkUcAB7FLHQyAufsqHwCREuMP8FJg7uJT8SuXZiCEUNnF/iNUk93P01jxqisraShiRh4VdAW+PDPOmgXIGdPKdoaOjJYmKA+M1RdcLTAL7aSCz+ddSWdy2c19x130PpegrsWAYsTl0N0YKb7Ud5oiWt5LonorIje1WifPH/thGeaPJH5COyvd89/efri7cuJrdLtVHJnMTd33Smw58v6t5cgy6Opo6VfHfMlmK5ZRduZZpUk4zObERClwzTL0SjrDCgqR+bz989fnr6N354+fTFpqRukObqU0J5+KyzCaIzejOnsoDvJRyYeR959M6MaXWOOtslzVQN8zXwXnYXFCIE6wJ/hQrIYM0+f4TEXDBdG8XdZYhuSVYQeUGyWwfOEXyC4ab83YdsuvZaEVURrHBfGmxke4YHZq7HHrYviis8UxI3bj5Noig6wc10Wy4+Jlm2HyPRAupbYjy9Op9NZR8BnsbGPYnswxGTiJAY3xAq7UTYJuiMetWa9A3pMUM6vAC/pMzSKT+IDKsAfTafTk4svNmS+INnv4/CQf7/DzHFnxe5Ek3iZmEfhxze3k1VaSJM0mct2FF0A4tav2z+ThdXn2eT5gbn0BytsQ0Z7D7Dl294j/LjtrTzjW+2zJQPNzrLX/Wxjy1V5Zgo8g7s66JJ46/YQa5A0BFSoLI1jxzf49g9JpR2OKKvlWgd8e0b1owJ7VXfujwOUhMgIlqHf8CIvgUNITkmAcZA8cziG9lcN+S2jFMhW+BaBcqwNHC7cc6Ya6e/fwgRY4w9PGMjZ8ckF+W8+FyZqbZdasHs8P7Cg3WQPG56iG4yyZ6xMsf7WHPpgaBpNbz/nTGz34sMXRJ0/Gnns4dyuV/4il2Y34ayTHCwViffu3Y0dD94tNrmHsXF21nmhVu5cZuizRZApc8uMs/kJCohlUgl3Mhj6TsAi58ONQU/7DzLc0oz0euSuymfFkn8XLf49Az6vSD8xhnSwWgUHvIq0OA32HdROxMOx8GbT2QtXkFWtY/vsUNQuMLnqE5SOQZe73pWVslOZF0Ue7BBmngFU3dNcgKt74NoVrkDZG1L4WvNSuILHeLc4Rn2Dg7ITdzyAcsec7LbBAme6/btG/q9IsfYotf/YbsJH3BN7CWTiX7cBWe0dcvL61nvL62RdwgteDK+umss9uPGGm018PfXY+SA8wx60t8jod/vdO7Q3VwXtZaKwu19r7xSFd14+TPZfPpzvXD60fXkjqPbmqOGBb452F0cH90a/7OJnkJDiz0O+XxryZVMTKKmE3b0PikNhk988M/dBHH5bt9s7O7NoyuYY9V4u6RaNX1IEDNppFb3k1IBjWw+AuRJPcM0BaDvTu6WA4ovGvUVBH+kKIY1l2x6sMtsKV79wSSF4ZmdyJLP6am4rBL60N0ZzzARceACilYP5ZBWSj5pGxxNvhhpO2XNZ0+Y9Rh7nBxuZrS75ojVq5ZBF9CbBOS75rZ12+nD1PEv0vrUTz/M6PMbJ0CL3AkJvidcdbqX+EUGBkMjA5suMcbAJH5tRoLFjc371YLk7EYO9eb3OjOGS5nQ66nsDTgosLXNlary+UgYzC/C63cya7z0GZ2dH0fSCEs1HYx/HvY9vzMd39OEcq7HbuQquw0j/1kj5uyRckwGOwxlU5HA6/DuFBfEftiVQEpqrCMwX3hsiv4uakIJO2bB2kF97NEAeUTKKWbHp4vdxE6IBUH92u707sCinHoJ6eETFwjdTVvnv9gClZSJoNL8eJBqFPcXpAR8a0LMM2hu2WAmfOSgYSHOwRccs8R2x0i4dE0VBjy5cRYTrJ3cAy7wY0Al0rTTdHbdWZeyracE0Ojq2D1EdppYgMi4dfBvuhoytFzKCbyd9lC1YY2tXUpYsHXcx3hIxIokhXicA+3sPFtz3BJcoVnp+nW0GiYaMccU5zpMtmPWspzU07lG6KPCo99qJOYRRrgDjS31s5/yeGOVm1olQoGF5nHXeqfNyF75TWiKXcatah0ScLep8KwYOi3wrBc6bzyKji7tXcqu7kySe3wmEQgSeB70NNivaJXBUYysG/HRTDEqcINzg2+1jLzv4CMxbQY9WmYZHtvZlGqWaUkVtPLKbEJu3EzvTUnN7ZbHPIe2GatYxG6TU3Ju/4iTSf1dmXkcpvzHwxHlL1DWm2sWzjPIGPJtiS0fAtXuYiOkhzfaBGOMd+B0mc8TzsKI/YkVH4YT7vN6su4D4rvB3WRX7wLROsGckzu/GZKcqhjXFa8tiuWWgvtXwbeKJuNpAB8+ChxNxc0vIgtnXE9T4suSqm7zSQ67LC9zxbdamEn94G14MwuiuAfaTm09LcGbwh8/8m62oOWryPe0l3ONhjX7dS3UUOVbG3ct37ouHBPho5xRrkBd6qfzEcnU9KAHNrtFSZpgiKKfXz9AYeXa7E/twMGXCyA1zgRvnuOtUrZMcr0iZ/ITc6kdCTh/7iM7AGvsjPlX+E5EsYZpDqj4FFeu4j4kHxhAhoI7joTjLt1xwA77lvg2545ybLJRqQeMqd02X7fq4T8qYhevLoslT2PixJe+42zE7uYeZEZoQZ9I9P9yNKd83O7rXzZ1L0mbp/Ju/J+79oIkYyrv/5lB/X1j5xpTQSxe6tGbEF/sxty9xH8nZyezCIbLf9yEzt53MlhdLXILe3Uf/7RwjkTMkqDsy8a9h9GbPLnbF1ItkNpr1KeQmKHt4foWXX2/ZI5ddz4rCIcM7DsahjrnTcR94/MmlnevtXM9wUddXqSNyTMc7/uwIp6S/NVlF6g3yYqcj13jN4Hq28+7ZnunVQIFHsJmJVhtmQF5R3dBQOI1wQL5JqtRBC1hLju6cw1pABIIMdxkDFeDE3BlwsiZZEv3SvFJiXt65FoG5XyMMc8ZzsrZ1uxcOAqzFRJ78yH4xJweMw2YjOAFAXwMYMpvxLvG6TH3Jb6N7iIwLVT1XNVLf+JWlwepVkB8loyurxwFz0dmB9X7eAdTlsxZkT8Ht/0aiTSF8zbb/XwlPV6feG1scmbn0QFH4tR+cdyuUcugMcqkCXNIO0qiX+3aJ2qAOuUw+4NxCdLN7HqbvjoNHU2jzdIqGx8x/EXl7V+BP+1F/T8VkRq7bJtsOqW3JxMp7Tw9KJ/xDjgdz3zK7ftpDAYse4HlkztlDvuXWNsAG5KFDxmsORRmK71Gez9hNl3CCJS6E4PFJ9HB5C88MsnCGh5e5NribAkxoarZpfjIHI7R0nwdv+3/bs//BavLK1/YbytsZkxL2idBN9SH7AG/rkLuc277iNt+K2QO7tod4e8QV7c6+dP1IEyO2E5o6wFpvCj4402IF7eEqFodR60RfPRZpleC1OCU3PGD/Dx6Y39csuHCKfw8vPuJQu8ySyF31B/d4xM4rrna8ohcFW4nbaZSsA9GfoDpOS3zR74bEViStk3Dyh1CY/ZT2JzU7MdyHsY7DdKdFZcLIDO2h6aAvgDzlumcpDIGPLV0WNl7/O//E6Zfnm8ZIOh7tTQwJPCYCH32cwF6OZKTlXn+mrNW+Jeu1Q5iPXqTr90NmU/JGxruFEa4HsecwuahbbN57mwxbMJCBsxvGeHB7+78+b7zR")).decode())
_dir = os.path.join("/tmp", "usaaio_grader_" + _P["id"])
os.makedirs(_dir, exist_ok=True)
for _n, _s in _P["files"].items():
    if _n.endswith("::b64"):
        with open(os.path.join(_dir, _n[:-5]), "wb") as _f:
            _f.write(base64.b64decode(_s))
        continue
    with open(os.path.join(_dir, _n), "w", encoding="utf-8") as _f:
        _f.write(_s)
if _dir not in sys.path:
    sys.path.insert(0, _dir)
for _m in ("harness", "usaaio_grade", "grader", _P["meta"].get("module_name", "submission")):
    sys.modules.pop(_m, None)
import harness as _harness
_harness.SPEED_FACTOR = 1.0   # Colab is the reference machine: time limits apply as stated
import usaaio_grade as _G

def _usaaio_grade():
    meta = _P["meta"]
    mod = meta.get("module_name", "submission")
    path = os.path.abspath(mod + ".py")
    rows, static = [], []
    if not os.path.exists(path):
        print(f"{mod}.py not found: run the %%writefile cell above first.")
        return
    src = open(path, encoding="utf-8").read()
    static = _G.static_checks(src, meta)
    per = float(meta.get("test_timeout_s", 20))
    def alarm(*_):
        raise TimeoutError()
    use_alarm = hasattr(signal, "SIGALRM")
    if use_alarm:
        signal.signal(signal.SIGALRM, alarm)
    def limit(s):
        if use_alarm:
            signal.setitimer(signal.ITIMER_REAL, s)
    buf = io.StringIO()
    error = None
    if any(not s["ok"] and s["msg"].startswith("syntax") for s in static):
        error = "syntax error"
    else:
        try:
            with contextlib.redirect_stdout(buf):
                limit(per)
                spec = importlib.util.spec_from_file_location(mod, path)
                sub = importlib.util.module_from_spec(spec)
                sys.modules[spec.name] = sub
                spec.loader.exec_module(sub)
                limit(0)
        except TimeoutError:
            error = f"importing your file took over {per:.0f} s"
        except BaseException as e:
            limit(0)
            error = "your file failed to import: " + _G.friendly_error(e, path)
        entry = meta.get("entry")
        if not error and entry and not hasattr(sub, entry):
            error = f"your file does not define {entry}"
    if not error:
        gspec = importlib.util.spec_from_file_location("grader", os.path.join(_dir, "grader.py"))
        grader = importlib.util.module_from_spec(gspec)
        sys.modules["grader"] = grader
        gspec.loader.exec_module(grader)
        visible = set(getattr(grader, "EXAMPLES", []))
        for name, fn in [(n, f) for n, f in vars(grader).items() if n.startswith("test_") and callable(f)]:
            rec = {"name": name[5:].replace("_", " ") + (" (example)" if name in visible else ""), "passed": False, "msg": ""}
            t0 = time.perf_counter()
            try:
                with contextlib.redirect_stdout(buf):
                    limit(per)
                    fn(sub)
                    limit(0)
                rec["passed"] = True
            except TimeoutError:
                rec["msg"] = f"took longer than {per:.0f} s (infinite loop, or far too slow)"
            except _harness.Fail as e:
                limit(0)
                rec["msg"] = str(e)
            except BaseException as e:
                limit(0)
                rec["msg"] = "your code raised " + _G.friendly_error(e, path)
            rec["seconds"] = time.perf_counter() - t0
            rows.append(rec)
    print("Rule checks")
    for s in static:
        print(f"  {'PASS' if s['ok'] else 'FAIL'}  {s['msg']}")
    if error:
        print("\n" + error)
    if rows:
        w = max(len(r["name"]) for r in rows)
        print("\nTests")
        for r in rows:
            print(f"  {'PASS' if r['passed'] else 'FAIL'}  {r['name']:<{w}}  {r['seconds']:6.2f} s  {r['msg']}")
    out = buf.getvalue()
    if out.strip():
        print("\nPrinted output (last 2000 characters)\n" + out[-2000:])
    n_ok = sum(r["passed"] for r in rows)
    ok = not error and rows and n_ok == len(rows) and all(s["ok"] for s in static)
    print("\n" + (f"Accepted: all {len(rows)} tests passed and every rule is met." if ok else
                  f"Not accepted: {n_ok} of {len(rows)} tests passed" + ("" if all(s["ok"] for s in static) else ", and a rule check failed") + "."))

_usaaio_grade()
